# 6.10 分塊邊界可以跟語言模型一起學嗎？


`catcatdogdog` 有 12 個英文字母，每個字母都是 1 個 UTF-8 byte。[6.6](https://birdhackor.github.io/tiny-perceptron-vlm/6.6.html) 的 `ByteTokenizer` 會把它編成 12 個普通 token，交給模型逐 byte 預測。本課其實已經在直接讀 bytes；Unicode 與 UTF-8 是文字的表示規則，沒有要求一定要再合併成詞片。這裡的新問題是：如果較大的模型不必在每個 byte 位置都工作，該選哪些位置來處理？

[6.3](https://birdhackor.github.io/tiny-perceptron-vlm/6.3.html) 的 byte-level BPE 從 bytes 學出一張固定的**子詞**（subword）字表，收錄常見的文字片段，讓大模型處理較少的 token。[6.9](https://birdhackor.github.io/tiny-perceptron-vlm/6.9.html) 已看到，同一套 BPE 遇到不同內容或讀入方式，實際切點仍可能不同；固定的是學好後的字表與合併規則，不是每個 token 都一樣長。另一條路則是保留 byte 輸入，讓較小的局部網路先讀取細節，形成較短的區塊特徵序列，再交給較大的主要模型。這種區塊常叫 **patch**：它由網路產生特徵，不需要先替每種區塊字串登記一個字表 ID。

先把邊界的作用看清楚。同一句可以每 4 bytes 分一塊，得到 `catc／atdo／gdog`；也可以分成 `cat／catdog／dog`，長度是 3、6、3。兩者都是三塊，主要模型要處理的都是三個位置，放在一起的內容卻不同。例如前文若已讓一段詞尾很容易猜，局部網路可能足以接完；遇到下一段較難預測的內容時，再讓主要模型處理新的資訊。切點就是在安排這種分工。下面兩種切法是手工示意，沒有量測哪種比較好；下半圖則追蹤已讀 bytes 如何變成區塊特徵，以及仍留在 byte 層的工作。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rewrite-06-10-learned-boundaries.svg")))

圖中的主要模型只處理三個區塊位置，局部網路仍要讀原來的 12 bytes，並利用區塊資訊預測後續 byte；右側箭頭保留局部細節。這不是 [6.8](https://birdhackor.github.io/tiny-perceptron-vlm/6.8.html) 把資料切成訓練視窗，也不是 6.9 的讀檔分段，而是在模型裡改變表示與計算分工。位置數不能直接當成程式呼叫次數或加速倍數，實際成本還包含局部網路與邊界判斷。

邊界可以根據預測訊號決定。**BLT**（Byte Latent Transformer，一種以 bytes 為輸入的階層模型）另外訓練一個較小的 byte 語言模型，估計「看到目前前文後，下一個 byte 有多不確定」，再用門檻等規則決定切點。容易預測的部分可以形成較長的區塊，把較多主要模型的位置留給不確定的地方。這裡的不確定性來自模型，切點仍由另外設定的規則決定。[BLT 原論文 §2.3–2.4](https://arxiv.org/html/2412.09871v1) 說明了這個做法。

**H-Net**（Hierarchical Network，階層網路）再把決定邊界的機制放進整個模型，和預測任務共同訓練。它從已讀內容的特徵計算邊界分數，決定哪些位置的特徵交給主要網路；預測代價的回饋會影響這套機制，而設計者指定的壓縮目標也約束區塊粒度。切得太細，主要網路仍要處理很多位置；太粗，又可能不利於預測所需的資訊交換。共同學習發生在訓練時，使用時參數可以固定，仍依不同輸入算出不同切點。[H-Net 原論文 §2.2、§2.4](https://arxiv.org/html/2507.07955v2) 介紹了這條路線。

這些研究常稱為 **tokenizer-free**，主要是省去外部預先學好的固定子詞字表，仍然有 bytes、邊界與模型內部的表示。依預測訊號切分，與讓切分策略一起學，是不同選擇；「動態分割」也不只是區塊長短不同。邊界要服務預測與計算預算，不必完全對齊人認為的詞。這是近年的一條研究方向，不能只憑上面的示意就判定已勝過 BPE。

練習看圖回答：兩種切法同樣都是三塊，為什麼仍值得比較切在哪裡？如果只選切在完整詞之間的方案，足夠嗎？答案要同時考慮哪些內容交給同一個區塊表示、局部與主要模型各做什麼，以及目前還沒量到的預測品質與總成本；只數區塊，或只看像不像詞，都還不能替這個設計作決定。
